**Alternate version that uses a helper LLM to summarize chunks during indexing, see original notebook first.**

## RAG: Adam & Isaac

Below, we take a [sample employee handbook](https://www.501commons.org/resources/tools-and-best-practices/human-resources/sample-employee-handbook-national-council-of-nonprofits) from the web and prepare it for use in a RAG application. We're using a Small-to-Big approach, where each embedding represents one paragraph, but is associated with (and will return) that paragraph and the ones preceding and following it.

The process should be transferable to other documents with some minor changes to the logic for identifying section and subsection headers.

In [1]:
!pip install duckdb FlagEmbedding pymupdf4llm httpx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.0/149.0 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 45.3 MB/s eta 0:00:00
  Created wheel for FlagEmbedding: filename=FlagEmbedding-1.3.5-py3-none-any.whl size=233746 sha256=2df56f87b23311d058e99f2fbb331550fce39bef03ade26e8c0ad83ab1362fb2
  Stored in directory: /root/.cache/pip/wheels/b2/1f/f6/78f862bb80cb959cc9960b7c4e2d1f702b1bc0e79d19b5f124
  Created wheel for warc3-wet-clueweb09: filename=warc3_wet_clueweb09-0.2.5-py3-none-any.whl size=18919 sha256=d6c78a6061a987ea8d

First, our imports. After importing `duckdb`, we open a new connection and activate its vector search extension.

In [2]:
from pathlib import Path
import re

import duckdb
from FlagEmbedding import BGEM3FlagModel
import httpx
import numpy as np
import pymupdf4llm
import torch

con = duckdb.connect()
con.install_extension("vss")
con.load_extension("vss")

sql="""SET GLOBAL hnsw_enable_experimental_persistence = true;"""

con.execute(sql)


In [3]:
# Get the handbook, if we don't already have it.
if not Path('handbook.pdf').exists():
    pdfreq = httpx.get('https://www.501commons.org/resources/tools-and-best-practices/human-resources/sample-employee-handbook-national-council-of-nonprofits', verify=False)
    pdf = pdfreq.content
    with open('handbook.pdf', 'wb') as f:
        f.write(pdf)

Below, we divide the handbook into paragraphs, then parse it into a list of dictionaries. We keep track of the section and subsection each paragraph is in, both for the next step and for the LLM's benefit during generation.

In [4]:
# Tried `pdfplumber`, but `pymupdf4llm` works better here.
# `ignore_graphics` keeps it from interpreting random lines as tables.
handbook = pymupdf4llm.to_markdown('handbook.pdf', table_strategy='lines_strict', ignore_graphics=True)

In [5]:
# Get rid of the table of contents and the (empty) mission statement.
handbook = handbook.split('I. MISSION')[2]

In [6]:
# Divide into paragraphs
handbook = handbook.split('\n\n')

**Modified from original division, all text within a section is kept together to avoid problems with lists.**

In [25]:
section = ''
subsection = ''
paragraphs = []
sections = 0

for para in handbook:
    para = para.strip()
    if re.match(r'[IVX]+\. [\w]+', para):
        para = re.sub(r'[IVX]+\. ', '', para)
        section = para
        subsection = ''
        sections += 1
    elif re.match(r'[A-Z]\. [\w]+', para):
        para = re.sub(r'[A-Z]\. ', '', para)
        subsection = para
    elif re.match(r'\[\d+\s?\]', para): # Page numbers. ignore.
        ...
    elif para.strip() == '':
        ...
    else:
      if len(paragraphs) < sections:
          paragraphs.append({
              'section': section,
              'subsection': subsection,
              'paragraph': ''
          })
      paragraphs[sections - 1]['paragraph'] += re.sub(r'\s+', ' ', para)

Print the first few paragraphs to make sure they look like what we expect.

In [27]:
for i in range(10):
  print(paragraphs[i])

{'section': 'OVERVIEW', 'subsection': '', 'paragraph': 'The {ORGANIZATION NAME} Employee Handbook (the “Handbook”) has been developed to provide general guidelines about {ORGANIZATION NAME} policies and procedures for employees. It is a guide to assist you in becoming familiar with some of the privileges and obligations of your employment, including {ORGANIZATION NAME}ʹs policy of voluntary at‐will employment. None of the policies or guidelines in the Handbook are intended to give rise to contractual rights or obligations, or to be construed as a guarantee of employment for any specific period of time, or any specific type of work. Additionally, with the exception of the voluntary at‐will employment policy, these guidelines are subject to modification, amendment or revocation by {ORGANIZATION NAME} at any time, without advance notice.The personnel polices of {ORGANIZATION NAME} are established by the Board of Directors, which has delegated authority and responsibility for their adminis

Import pretrained LLM for text summarization.

In [45]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
model_name = "sshleifer/distilbart-cnn-12-6"
tokenizer = AutoTokenizer.from_pretrained(model_name)
summarizer = AutoModelForSeq2SeqLM.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

In [46]:
def summarize_chunk(text, max_new_tokens=100):
  inputs = tokenizer("summarize: " + text, return_tensors="pt", truncation=True)
  summary_ids = summarizer.generate(**inputs, max_new_tokens=max_new_tokens)
  return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

In [49]:
summaries = []
for i in range(len(paragraphs)):
  summaries.append(summarize_chunk(paragraphs[i]['paragraph']))

Not the best summarization considering some cutoffs and mispellings. Some information is probably lost as well considering the ratio of size between chunks and paragraphs.

In [50]:
summaries

[' The Handbook is a guide to become familiar with some of the privileges and obligations of your employment . None of the policies or guidelines in the Handbook are intended to give rise to contractual rights or obligations . With the exception of the voluntary at‐will employment policy, these guidelines are subject to modification, amendment or revocation .',
 ' All employment at {ORGANIZATION NAME} is “at‐will” That means that employees may be terminated from employment with or without cause . Employees are free to leave the employment of {ORGNATION NAME}. Any representation by any . officer or employee contrary to this policy is not binding upon {ORIZATION .',
 ' The Board of Directors and Executive Director of {ORGANIZATION NAME] will not discriminate against any employee or applicant in a manner that violates the law . Each person is evaluated on the basis of personal skill and merit . All employment practices and activities are conducted on a non-discriminatory basis .',
 ' Empl

In [52]:
#apply summaries
for i in range(len(paragraphs)):
  paragraphs[i]['paragraph'] = summaries[i]

Load an embedding model. Use the GPU if we have one.

In [51]:
device = "cpu"
# use a GPU if available to speed up the embedding computation
if torch.cuda.is_available(): device = "cuda" # Nvidia GPU
elif torch.backends.mps.is_available(): device = "mps" # Apple silicon GPU

model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True, device=device)

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

...

onnx/model.onnx_data:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

onnx/tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

sparse_linear.pt:   0%|          | 0.00/3.52k [00:00<?, ?B/s]

Create a database table to store our data and embeddings, then add the data to it.

In [56]:
con.execute("CREATE SEQUENCE IF NOT EXISTS seq_handbookid START 1")
con.execute("DROP TABLE IF EXISTS handbook")
qry = """CREATE TABLE handbook
(
  id INTEGER PRIMARY KEY DEFAULT NEXTVAL('seq_handbookid'),
  section TEXT,
  subsection TEXT,
  paragraph TEXT,
  embedding FLOAT[1024]
)

"""

con.execute(qry)

In [57]:
for para in paragraphs:
    embedding = model.encode(para['paragraph'])["dense_vecs"]
    qry = f"""INSERT INTO handbook (section, subsection, paragraph, embedding) VALUES (?, ?, ?, ?)"""
    con.execute(qry, (para['section'], para['subsection'], para['paragraph'], embedding))

Make sure the `handbook` table has the content we expect.

In [58]:
con.execute('SELECT * FROM handbook LIMIT 5').fetch_df()

,id,section,subsection,paragraph,embedding
0,1,OVERVIEW,,The Handbook is a guide to become familiar wi...,"[-0.044952393, -0.0008196831, -0.03363037, -0...."
1,2,VOLUNTARY AT‐WILL EMPLOYMENT,,All employment at {ORGANIZATION NAME} is “at‐...,"[-0.005630493, -0.0007557869, -0.020050049, 0...."
2,3,EQUAL EMPLOYMENT OPPORTUNITY,,The Board of Directors and Executive Director...,"[-0.031677246, -0.02835083, -0.014945984, 0.00..."
3,4,POLICY AGAINST WORKPLACE HARASSMENT,,Employees are expected to conduct themselves ...,"[-0.022659302, 0.0012140274, -0.027450562, 0.0..."
4,5,SOLICITATION,,“Work time” includes time spent in actual per...,"[-0.034851074, -0.0068130493, -0.026412964, 0...."


Here we create an `embed()` function and add it to the database engine as a UDF. This will simplify querying the database later.

In [59]:
from duckdb.typing import VARCHAR

def embed(sentence: str) -> np.ndarray:
    return model.encode(sentence)['dense_vecs']

con.create_function("embed", embed, [VARCHAR], 'FLOAT[1024]')


qry = "SELECT embed('How much can I drink at work?') AS query_embedding;"
con.execute(qry).fetch_df()

,query_embedding
0,"[-0.017196655, 0.012336731, -0.041107178, -0.0..."


Here, create a simple search function for testing, to see if the naive search results look sensible.

In [60]:
def search(q: str):
    return con.execute("""
        FROM handbook
        SELECT section, subsection, paragraph, array_inner_product(embedding, embed($q)) AS similarity
        ORDER BY similarity DESC
        LIMIT 3""",
        {"q": q}
    ).fetch_df()

drinking = search('How much can i drink at work?')
drinking

,section,subsection,paragraph,similarity
0,"HOURS OF WORK, ATTENDANCE AND PUNCTUALITY",Hours of Work,The normal work week for {ORGANIZATION NAME} ...,0.537630
1,SOLICITATION,,“Work time” includes time spent in actual per...,0.526055
2,POLICY AGAINST WORKPLACE HARASSMENT,,Employees are expected to conduct themselves ...,0.515226


In [62]:
print(drinking['paragraph'][0])

 The normal work week for {ORGANIZATION NAME} shall consist of five (5), seven (7) hour days . Punctuality and regular attendance are expected of all employees . Excessive absences (whether excused or unexcused), tardiness or leaving early is unacceptable .


In [63]:
search('How do i apply for travel expense reimbursement?')['paragraph'][0]

' Reimbursement is authorized for reasonable and necessary expenses incurred in carrying out job responsibilities . Transportation costs are paid by {ORGANIZATION NAME} for work outside normal work hours if the employee is on official business . Employees may also request a travel advance to cover anticipated expenses approved travel .'

### Limitations

* The drinking example only returns a few items from the middle of an itemized list, showing that sometimes the context may not contain all the context the LLM needs. We could attempt to address that with special logic for lists (the paragraph before that list would help), but a better solution might be to let an LLM chunk the document for us and add necessary context to each item.
* There's probably only one useful context for a lot of likely queries to this dataset; an extra layer of LLM-driven relevance checking might help.


###Addendum
Using a helper LLM can help during the indexing process to capture broader context while keeping chunks small. Our basic implementation appears to have some obvious issues with fully summarizing the chunks, and may completely omit the data relevant to the query. Splitting the chunks in a smarter manner rather than attempting to summarize entire sections would likely be help. Further experimentation with the helper LLM to yield a better chunk summarization could be done. The few models tried during implementation were rather small, but since they're only used for the initial indexing, larger ones could easily be used without impacting retrieval.